In [117]:
import pandas as pd
import re
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# NMI flights

In [118]:
df_nmi = pd.read_csv("csv_data/google_flight_nmi.csv")
df_nmi.head()

,Unnamed: 0,Departure_Time,Arrival_Time,Airline,Duration,Route,Stops,Layover_Time_Location,CO2_Emissions,Emissions_Change,Price,Class
0,0,8:00 AM,2:50 PM,IndiGo,6 hr 50 min,NMI–BLR,1 stop,3 hr 50 min KLH,88 kg CO2e,+28% emissions,"₹6,073",economy
1,1,11:05 AM,8:55 PM,IndiGo,9 hr 50 min,NMI–BLR,1 stop,6 hr 50 min GOI,92 kg CO2e,+33% emissions,"₹6,937",economy
2,2,11:05 AM,3:45 PM,IndiGo,4 hr 40 min,NMI–BLR,1 stop,1 hr 50 min GOI,92 kg CO2e,+33% emissions,"₹8,086",economy
3,17,7:15 PM,11:00 PM,IndiGo,3 hr 45 min,BLR–NMI,1 stop,1 hr 20 min GOI,91 kg CO2e,+32% emissions,"₹6,005",economy
4,18,5:40 PM,11:00 PM,IndiGo,5 hr 20 min,BLR–NMI,1 stop,2 hr 50 min GOI,91 kg CO2e,+32% emissions,"₹6,005",economy


#### Data preprocessing for EDA:

##### Remove ₹ and , symbol from Price

In [119]:
df_nmi["Price"] = (
        df_nmi["Price"]
        .str.replace("₹", "", regex=False)
        .str.replace(",", "", regex=False)
    )
df_nmi["Price"] = pd.to_numeric(
        df_nmi["Price"],
        errors="coerce"
    )
df_nmi.head()

,Unnamed: 0,Departure_Time,Arrival_Time,Airline,Duration,Route,Stops,Layover_Time_Location,CO2_Emissions,Emissions_Change,Price,Class
0,0,8:00 AM,2:50 PM,IndiGo,6 hr 50 min,NMI–BLR,1 stop,3 hr 50 min KLH,88 kg CO2e,+28% emissions,6073,economy
1,1,11:05 AM,8:55 PM,IndiGo,9 hr 50 min,NMI–BLR,1 stop,6 hr 50 min GOI,92 kg CO2e,+33% emissions,6937,economy
2,2,11:05 AM,3:45 PM,IndiGo,4 hr 40 min,NMI–BLR,1 stop,1 hr 50 min GOI,92 kg CO2e,+33% emissions,8086,economy
3,17,7:15 PM,11:00 PM,IndiGo,3 hr 45 min,BLR–NMI,1 stop,1 hr 20 min GOI,91 kg CO2e,+32% emissions,6005,economy
4,18,5:40 PM,11:00 PM,IndiGo,5 hr 20 min,BLR–NMI,1 stop,2 hr 50 min GOI,91 kg CO2e,+32% emissions,6005,economy


##### Dividing Route into Source and Destination

In [120]:
df_nmi[["Source", "Destination"]] = (
        df_nmi["Route"]
        .str.split("–", expand=True)
    )
df_nmi.head()

,Unnamed: 0,Departure_Time,Arrival_Time,Airline,Duration,Route,Stops,Layover_Time_Location,CO2_Emissions,Emissions_Change,Price,Class,Source,Destination
0,0,8:00 AM,2:50 PM,IndiGo,6 hr 50 min,NMI–BLR,1 stop,3 hr 50 min KLH,88 kg CO2e,+28% emissions,6073,economy,NMI,BLR
1,1,11:05 AM,8:55 PM,IndiGo,9 hr 50 min,NMI–BLR,1 stop,6 hr 50 min GOI,92 kg CO2e,+33% emissions,6937,economy,NMI,BLR
2,2,11:05 AM,3:45 PM,IndiGo,4 hr 40 min,NMI–BLR,1 stop,1 hr 50 min GOI,92 kg CO2e,+33% emissions,8086,economy,NMI,BLR
3,17,7:15 PM,11:00 PM,IndiGo,3 hr 45 min,BLR–NMI,1 stop,1 hr 20 min GOI,91 kg CO2e,+32% emissions,6005,economy,BLR,NMI
4,18,5:40 PM,11:00 PM,IndiGo,5 hr 20 min,BLR–NMI,1 stop,2 hr 50 min GOI,91 kg CO2e,+32% emissions,6005,economy,BLR,NMI


##### Converting Duration into decimal

In [121]:
duration = (
        df_nmi["Duration"]
        .str.replace("hr", "hours")
        .str.replace("min", "minutes")
    )

df_nmi["Duration"] = (
        pd.to_timedelta(
            duration,
            errors="coerce"
        )
        .dt.total_seconds()
        / 3600
    )

df_nmi["Duration"] = (
        df_nmi["Duration"]
        .round(2)
    )
df_nmi.head()

,Unnamed: 0,Departure_Time,Arrival_Time,Airline,Duration,Route,Stops,Layover_Time_Location,CO2_Emissions,Emissions_Change,Price,Class,Source,Destination
0,0,8:00 AM,2:50 PM,IndiGo,6.83,NMI–BLR,1 stop,3 hr 50 min KLH,88 kg CO2e,+28% emissions,6073,economy,NMI,BLR
1,1,11:05 AM,8:55 PM,IndiGo,9.83,NMI–BLR,1 stop,6 hr 50 min GOI,92 kg CO2e,+33% emissions,6937,economy,NMI,BLR
2,2,11:05 AM,3:45 PM,IndiGo,4.67,NMI–BLR,1 stop,1 hr 50 min GOI,92 kg CO2e,+33% emissions,8086,economy,NMI,BLR
3,17,7:15 PM,11:00 PM,IndiGo,3.75,BLR–NMI,1 stop,1 hr 20 min GOI,91 kg CO2e,+32% emissions,6005,economy,BLR,NMI
4,18,5:40 PM,11:00 PM,IndiGo,5.33,BLR–NMI,1 stop,2 hr 50 min GOI,91 kg CO2e,+32% emissions,6005,economy,BLR,NMI


##### CO2_Emissions to decimal

In [122]:
df_nmi["CO2_Emissions"] = (
        df_nmi["CO2_Emissions"]
        .str.extract(r"(\d+)")
        .iloc[:, 0]
    )

df_nmi["CO2_Emissions"] = (
        pd.to_numeric(
            df_nmi["CO2_Emissions"],
            errors="coerce"
        )
    )
df_nmi.head()


,Unnamed: 0,Departure_Time,Arrival_Time,Airline,Duration,Route,Stops,Layover_Time_Location,CO2_Emissions,Emissions_Change,Price,Class,Source,Destination
0,0,8:00 AM,2:50 PM,IndiGo,6.83,NMI–BLR,1 stop,3 hr 50 min KLH,88,+28% emissions,6073,economy,NMI,BLR
1,1,11:05 AM,8:55 PM,IndiGo,9.83,NMI–BLR,1 stop,6 hr 50 min GOI,92,+33% emissions,6937,economy,NMI,BLR
2,2,11:05 AM,3:45 PM,IndiGo,4.67,NMI–BLR,1 stop,1 hr 50 min GOI,92,+33% emissions,8086,economy,NMI,BLR
3,17,7:15 PM,11:00 PM,IndiGo,3.75,BLR–NMI,1 stop,1 hr 20 min GOI,91,+32% emissions,6005,economy,BLR,NMI
4,18,5:40 PM,11:00 PM,IndiGo,5.33,BLR–NMI,1 stop,2 hr 50 min GOI,91,+32% emissions,6005,economy,BLR,NMI


##### Emissions change to decimal

In [123]:
df_nmi["Emissions_Change"] = (
        df_nmi["Emissions_Change"]
        .str.extract(r"([+-]?\d+)")
        .iloc[:, 0]
    )

df_nmi["Emissions_Change"] = (
        pd.to_numeric(
            df_nmi["Emissions_Change"],
            errors="coerce"
        )
    )
df_nmi.head()

,Unnamed: 0,Departure_Time,Arrival_Time,Airline,Duration,Route,Stops,Layover_Time_Location,CO2_Emissions,Emissions_Change,Price,Class,Source,Destination
0,0,8:00 AM,2:50 PM,IndiGo,6.83,NMI–BLR,1 stop,3 hr 50 min KLH,88,28.0,6073,economy,NMI,BLR
1,1,11:05 AM,8:55 PM,IndiGo,9.83,NMI–BLR,1 stop,6 hr 50 min GOI,92,33.0,6937,economy,NMI,BLR
2,2,11:05 AM,3:45 PM,IndiGo,4.67,NMI–BLR,1 stop,1 hr 50 min GOI,92,33.0,8086,economy,NMI,BLR
3,17,7:15 PM,11:00 PM,IndiGo,3.75,BLR–NMI,1 stop,1 hr 20 min GOI,91,32.0,6005,economy,BLR,NMI
4,18,5:40 PM,11:00 PM,IndiGo,5.33,BLR–NMI,1 stop,2 hr 50 min GOI,91,32.0,6005,economy,BLR,NMI


#### Number of Stops to int

In [124]:
print(df_nmi["Stops"].unique())
df_nmi["Stops"] = df_nmi["Stops"].map({
    "Nonstop": 0,
    "1 stop": 1,
    "2 stops": 2
})
df_nmi.head()

['1 stop' 'Nonstop']


,Unnamed: 0,Departure_Time,Arrival_Time,Airline,Duration,Route,Stops,Layover_Time_Location,CO2_Emissions,Emissions_Change,Price,Class,Source,Destination
0,0,8:00 AM,2:50 PM,IndiGo,6.83,NMI–BLR,1,3 hr 50 min KLH,88,28.0,6073,economy,NMI,BLR
1,1,11:05 AM,8:55 PM,IndiGo,9.83,NMI–BLR,1,6 hr 50 min GOI,92,33.0,6937,economy,NMI,BLR
2,2,11:05 AM,3:45 PM,IndiGo,4.67,NMI–BLR,1,1 hr 50 min GOI,92,33.0,8086,economy,NMI,BLR
3,17,7:15 PM,11:00 PM,IndiGo,3.75,BLR–NMI,1,1 hr 20 min GOI,91,32.0,6005,economy,BLR,NMI
4,18,5:40 PM,11:00 PM,IndiGo,5.33,BLR–NMI,1,2 hr 50 min GOI,91,32.0,6005,economy,BLR,NMI


##### Layover time and location into 2 columns

In [125]:
def split_layover(x):
    if pd.isna(x):
        return pd.Series([0.0, "NO_LOC"])

    # no layover - 0 minutes
    if "0 minute" in x.lower():
        return pd.Series([0.0, "NO_LOC"])

    # get hours and/or minutes
    hr = re.search(r"(\d+)\s*hr", x)
    mins = re.search(r"(\d+)\s*min", x)

    h = int(hr.group(1)) if hr else 0
    m = int(mins.group(1)) if mins else 0
    decimal = round(h+m/60, 2)

    # get location
    loc = re.search(r"\b([A-Z]{3})\b$", x)

    location = (
        loc.group(1)
        if loc
        else "NO_LOC"
    )

    return pd.Series([
        decimal,
        location
    ])


df_nmi[
    ["Layover_Time", "Layover_Location"]
] = (
    df_nmi["Layover_Time_Location"]
    .apply(split_layover)
)

df_nmi.drop(
    columns=["Layover_Time_Location"],
    inplace=True
)
df_nmi.head()

,Unnamed: 0,Departure_Time,Arrival_Time,Airline,Duration,Route,Stops,CO2_Emissions,Emissions_Change,Price,Class,Source,Destination,Layover_Time,Layover_Location
0,0,8:00 AM,2:50 PM,IndiGo,6.83,NMI–BLR,1,88,28.0,6073,economy,NMI,BLR,3.83,KLH
1,1,11:05 AM,8:55 PM,IndiGo,9.83,NMI–BLR,1,92,33.0,6937,economy,NMI,BLR,6.83,GOI
2,2,11:05 AM,3:45 PM,IndiGo,4.67,NMI–BLR,1,92,33.0,8086,economy,NMI,BLR,1.83,GOI
3,17,7:15 PM,11:00 PM,IndiGo,3.75,BLR–NMI,1,91,32.0,6005,economy,BLR,NMI,1.33,GOI
4,18,5:40 PM,11:00 PM,IndiGo,5.33,BLR–NMI,1,91,32.0,6005,economy,BLR,NMI,2.83,GOI


##### Departure and Arrival time to decimal

In [126]:
df_nmi["Departure_Time"] = df_nmi["Departure_Time"].str.strip()
df_nmi["Arrival_Time"] = df_nmi["Arrival_Time"].str.strip()
dep = pd.to_datetime(
        df_nmi["Departure_Time"],
        format="%I:%M %p",
        errors="coerce"
    )
arr = pd.to_datetime(
        df_nmi
["Arrival_Time"],
        format="%I:%M %p",
        errors="coerce"
    )
df_nmi["Departure_Hour"] = dep.dt.hour + dep.dt.minute / 60
df_nmi["Arrival_Hour"] = arr.dt.hour + arr.dt.minute / 60

df_nmi.head()

,Unnamed: 0,Departure_Time,Arrival_Time,Airline,Duration,Route,Stops,CO2_Emissions,Emissions_Change,Price,Class,Source,Destination,Layover_Time,Layover_Location,Departure_Hour,Arrival_Hour
0,0,8:00 AM,2:50 PM,IndiGo,6.83,NMI–BLR,1,88,28.0,6073,economy,NMI,BLR,3.83,KLH,8.000000,14.833333
1,1,11:05 AM,8:55 PM,IndiGo,9.83,NMI–BLR,1,92,33.0,6937,economy,NMI,BLR,6.83,GOI,11.083333,20.916667
2,2,11:05 AM,3:45 PM,IndiGo,4.67,NMI–BLR,1,92,33.0,8086,economy,NMI,BLR,1.83,GOI,11.083333,15.750000
3,17,7:15 PM,11:00 PM,IndiGo,3.75,BLR–NMI,1,91,32.0,6005,economy,BLR,NMI,1.33,GOI,19.250000,23.000000
4,18,5:40 PM,11:00 PM,IndiGo,5.33,BLR–NMI,1,91,32.0,6005,economy,BLR,NMI,2.83,GOI,17.666667,23.000000


##### Remove unwanted columns



In [127]:
df_nmi = df_nmi.drop(
        columns=[
            "Route",
            "Unnamed: 0",
            "Departure_Time","Arrival_Time"
        ],
        errors="ignore"
    )

### Data (after cleaning and transformation):

In [128]:
df_nmi.head()

,Airline,Duration,Stops,CO2_Emissions,Emissions_Change,Price,Class,Source,Destination,Layover_Time,Layover_Location,Departure_Hour,Arrival_Hour
0,IndiGo,6.83,1,88,28.0,6073,economy,NMI,BLR,3.83,KLH,8.000000,14.833333
1,IndiGo,9.83,1,92,33.0,6937,economy,NMI,BLR,6.83,GOI,11.083333,20.916667
2,IndiGo,4.67,1,92,33.0,8086,economy,NMI,BLR,1.83,GOI,11.083333,15.750000
3,IndiGo,3.75,1,91,32.0,6005,economy,BLR,NMI,1.33,GOI,19.250000,23.000000
4,IndiGo,5.33,1,91,32.0,6005,economy,BLR,NMI,2.83,GOI,17.666667,23.000000


#### Airlines:


In [129]:
print(df_nmi["Airline"].unique())

['IndiGo' 'Akasa Air']


##### Sources:


In [130]:
print(df_nmi["Source"].unique())

['NMI' 'BLR' 'MAA' 'DEL' 'CCU' 'COK' 'IXC' 'CJB' 'JAI' 'TRV' 'PAT' 'GOX'
 'IXE' 'HYD' 'GOI']


##### Destinations:

In [131]:
print(df_nmi["Destination"].unique())

['BLR' 'NMI' 'MAA' 'DEL' 'CCU' 'COK' 'IXC' 'CJB' 'JAI' 'TRV' 'IXR' 'PAT'
 'IXE' 'HYD' 'GOX' 'GOI']


#### Number of flights:


In [132]:
len(df_nmi)

98

#### Average price:

In [133]:
print(df_nmi["Price"].mean())

7193.877551020408


#### Average CO2 Emissions:

In [155]:
print(df_nmi["CO2_Emissions"].mean())

98.9795918367347


#### Number of flights to (and fro) each destination:

In [134]:
# Flights from NMI 
from_nmi = (
    df_nmi[df_nmi["Source"] == "NMI"]
    .groupby("Destination")
    .size()
    .rename("Flights from NMI")
)

# Flights to NMI 
to_nmi = (
    df_nmi[df_nmi["Destination"] == "NMI"]
    .groupby("Source")
    .size()
    .rename("Flights to NMI")
)

# Combine 
flights_nmi = pd.concat([to_nmi, from_nmi], axis=1).fillna(0).astype(int)

flights_nmi.index.name = "Airport"
flights_nmi = flights_nmi.reset_index()

print(flights_nmi)

   Airport  Flights to NMI  Flights from NMI
0      BLR               7                 5
1      CCU               3                 3
2      CJB               3                 5
3      COK               4                 4
4      DEL               6                 7
5      GOI               2                 2
6      GOX               2                 1
7      HYD               2                 2
8      IXC               6                 2
9      IXE               3                 3
10     JAI               2                 2
11     MAA               5                 4
12     PAT               4                 3
13     TRV               3                 2
14     IXR               0                 1


# BOM flights

In [135]:
df_bom = pd.read_csv("csv_data/google_flight_bom.csv")
df_bom.head()

,Unnamed: 0,Departure_Time,Arrival_Time,Airline,Duration,Route,Stops,Layover_Time_Location,CO2_Emissions,Emissions_Change,Price,Class
0,0,11:20 AM,3:45 PM,IndiGo,4 hr 25 min,BOM–BLR,1 stop,1 hr 55 min GOI,92 kg CO2e,+33% emissions,"₹4,383",economy
1,1,4:40 PM,10:15 PM,IndiGo,5 hr 35 min,BOM–BLR,1 stop,2 hr 50 min GOI,104 kg CO2e,+51% emissions,"₹4,383",economy
2,2,5:45 AM,10:55 AM,IndiGo,5 hr 10 min,BOM–BLR,1 stop,2 hr 15 min HYD,106 kg CO2e,+54% emissions,"₹4,514",economy
3,3,9:15 AM,1:35 PM,IndiGo,4 hr 20 min,BOM–BLR,1 stop,1 hr 45 min GOX,95 kg CO2e,+38% emissions,"₹4,514",economy
4,4,3:15 PM,7:05 PM,IndiGo,3 hr 50 min,BOM–BLR,1 stop,1 hr HYD,106 kg CO2e,+54% emissions,"₹4,514",economy


#### Data preprocessing for EDA-

##### Remove ₹ and , symbol from Price

In [136]:
df_bom["Price"] = (
        df_bom["Price"]
        .str.replace("₹", "", regex=False)
        .str.replace(",", "", regex=False)
    )
df_bom["Price"] = pd.to_numeric(
        df_bom["Price"],
        errors="coerce"
    )
df_bom.head()

,Unnamed: 0,Departure_Time,Arrival_Time,Airline,Duration,Route,Stops,Layover_Time_Location,CO2_Emissions,Emissions_Change,Price,Class
0,0,11:20 AM,3:45 PM,IndiGo,4 hr 25 min,BOM–BLR,1 stop,1 hr 55 min GOI,92 kg CO2e,+33% emissions,4383,economy
1,1,4:40 PM,10:15 PM,IndiGo,5 hr 35 min,BOM–BLR,1 stop,2 hr 50 min GOI,104 kg CO2e,+51% emissions,4383,economy
2,2,5:45 AM,10:55 AM,IndiGo,5 hr 10 min,BOM–BLR,1 stop,2 hr 15 min HYD,106 kg CO2e,+54% emissions,4514,economy
3,3,9:15 AM,1:35 PM,IndiGo,4 hr 20 min,BOM–BLR,1 stop,1 hr 45 min GOX,95 kg CO2e,+38% emissions,4514,economy
4,4,3:15 PM,7:05 PM,IndiGo,3 hr 50 min,BOM–BLR,1 stop,1 hr HYD,106 kg CO2e,+54% emissions,4514,economy


##### Dividing Route into Source and Destination

In [137]:
df_bom[["Source", "Destination"]] = (
        df_bom["Route"]
        .str.split("–", expand=True)
    )
df_bom.head()

,Unnamed: 0,Departure_Time,Arrival_Time,Airline,Duration,Route,Stops,Layover_Time_Location,CO2_Emissions,Emissions_Change,Price,Class,Source,Destination
0,0,11:20 AM,3:45 PM,IndiGo,4 hr 25 min,BOM–BLR,1 stop,1 hr 55 min GOI,92 kg CO2e,+33% emissions,4383,economy,BOM,BLR
1,1,4:40 PM,10:15 PM,IndiGo,5 hr 35 min,BOM–BLR,1 stop,2 hr 50 min GOI,104 kg CO2e,+51% emissions,4383,economy,BOM,BLR
2,2,5:45 AM,10:55 AM,IndiGo,5 hr 10 min,BOM–BLR,1 stop,2 hr 15 min HYD,106 kg CO2e,+54% emissions,4514,economy,BOM,BLR
3,3,9:15 AM,1:35 PM,IndiGo,4 hr 20 min,BOM–BLR,1 stop,1 hr 45 min GOX,95 kg CO2e,+38% emissions,4514,economy,BOM,BLR
4,4,3:15 PM,7:05 PM,IndiGo,3 hr 50 min,BOM–BLR,1 stop,1 hr HYD,106 kg CO2e,+54% emissions,4514,economy,BOM,BLR


##### Converting Duration into decimal

In [138]:
duration = (
        df_bom["Duration"]
        .str.replace("hr", "hours")
        .str.replace("min", "minutes")
    )

df_bom["Duration"] = (
        pd.to_timedelta(
            duration,
            errors="coerce"
        )
        .dt.total_seconds()
        / 3600
    )

df_bom["Duration"] = (
        df_nmi["Duration"]
        .round(2)
    )
df_bom.head()

,Unnamed: 0,Departure_Time,Arrival_Time,Airline,Duration,Route,Stops,Layover_Time_Location,CO2_Emissions,Emissions_Change,Price,Class,Source,Destination
0,0,11:20 AM,3:45 PM,IndiGo,6.83,BOM–BLR,1 stop,1 hr 55 min GOI,92 kg CO2e,+33% emissions,4383,economy,BOM,BLR
1,1,4:40 PM,10:15 PM,IndiGo,9.83,BOM–BLR,1 stop,2 hr 50 min GOI,104 kg CO2e,+51% emissions,4383,economy,BOM,BLR
2,2,5:45 AM,10:55 AM,IndiGo,4.67,BOM–BLR,1 stop,2 hr 15 min HYD,106 kg CO2e,+54% emissions,4514,economy,BOM,BLR
3,3,9:15 AM,1:35 PM,IndiGo,3.75,BOM–BLR,1 stop,1 hr 45 min GOX,95 kg CO2e,+38% emissions,4514,economy,BOM,BLR
4,4,3:15 PM,7:05 PM,IndiGo,5.33,BOM–BLR,1 stop,1 hr HYD,106 kg CO2e,+54% emissions,4514,economy,BOM,BLR


##### CO2_Emissions to decimal

In [139]:
df_bom["CO2_Emissions"] = (
        df_bom["CO2_Emissions"]
        .str.extract(r"(\d+)")
        .iloc[:, 0]
    )

df_bom["CO2_Emissions"] = (
        pd.to_numeric(
            df_bom["CO2_Emissions"],
            errors="coerce"
        )
    )
df_bom.head()


,Unnamed: 0,Departure_Time,Arrival_Time,Airline,Duration,Route,Stops,Layover_Time_Location,CO2_Emissions,Emissions_Change,Price,Class,Source,Destination
0,0,11:20 AM,3:45 PM,IndiGo,6.83,BOM–BLR,1 stop,1 hr 55 min GOI,92,+33% emissions,4383,economy,BOM,BLR
1,1,4:40 PM,10:15 PM,IndiGo,9.83,BOM–BLR,1 stop,2 hr 50 min GOI,104,+51% emissions,4383,economy,BOM,BLR
2,2,5:45 AM,10:55 AM,IndiGo,4.67,BOM–BLR,1 stop,2 hr 15 min HYD,106,+54% emissions,4514,economy,BOM,BLR
3,3,9:15 AM,1:35 PM,IndiGo,3.75,BOM–BLR,1 stop,1 hr 45 min GOX,95,+38% emissions,4514,economy,BOM,BLR
4,4,3:15 PM,7:05 PM,IndiGo,5.33,BOM–BLR,1 stop,1 hr HYD,106,+54% emissions,4514,economy,BOM,BLR


##### Emissions change to decimal

In [140]:
df_bom["Emissions_Change"] = (
        df_bom["Emissions_Change"]
        .str.extract(r"([+-]?\d+)")
        .iloc[:, 0]
    )

df_bom["Emissions_Change"] = (
        pd.to_numeric(
            df_bom["Emissions_Change"],
            errors="coerce"
        )
    )
df_bom.head()

,Unnamed: 0,Departure_Time,Arrival_Time,Airline,Duration,Route,Stops,Layover_Time_Location,CO2_Emissions,Emissions_Change,Price,Class,Source,Destination
0,0,11:20 AM,3:45 PM,IndiGo,6.83,BOM–BLR,1 stop,1 hr 55 min GOI,92,33.0,4383,economy,BOM,BLR
1,1,4:40 PM,10:15 PM,IndiGo,9.83,BOM–BLR,1 stop,2 hr 50 min GOI,104,51.0,4383,economy,BOM,BLR
2,2,5:45 AM,10:55 AM,IndiGo,4.67,BOM–BLR,1 stop,2 hr 15 min HYD,106,54.0,4514,economy,BOM,BLR
3,3,9:15 AM,1:35 PM,IndiGo,3.75,BOM–BLR,1 stop,1 hr 45 min GOX,95,38.0,4514,economy,BOM,BLR
4,4,3:15 PM,7:05 PM,IndiGo,5.33,BOM–BLR,1 stop,1 hr HYD,106,54.0,4514,economy,BOM,BLR


#### Number of Stops to int

In [141]:
print(df_bom["Stops"].unique())
df_bom["Stops"] = df_bom["Stops"].map({
    "Nonstop": 0,
    "1 stop": 1,
    "2 stops": 2
})
df_bom.head()

['1 stop' 'Nonstop']


,Unnamed: 0,Departure_Time,Arrival_Time,Airline,Duration,Route,Stops,Layover_Time_Location,CO2_Emissions,Emissions_Change,Price,Class,Source,Destination
0,0,11:20 AM,3:45 PM,IndiGo,6.83,BOM–BLR,1,1 hr 55 min GOI,92,33.0,4383,economy,BOM,BLR
1,1,4:40 PM,10:15 PM,IndiGo,9.83,BOM–BLR,1,2 hr 50 min GOI,104,51.0,4383,economy,BOM,BLR
2,2,5:45 AM,10:55 AM,IndiGo,4.67,BOM–BLR,1,2 hr 15 min HYD,106,54.0,4514,economy,BOM,BLR
3,3,9:15 AM,1:35 PM,IndiGo,3.75,BOM–BLR,1,1 hr 45 min GOX,95,38.0,4514,economy,BOM,BLR
4,4,3:15 PM,7:05 PM,IndiGo,5.33,BOM–BLR,1,1 hr HYD,106,54.0,4514,economy,BOM,BLR


##### Layover time and location into 2 columns

In [142]:
def split_layover(x):
    if pd.isna(x):
        return pd.Series([0.0, "NO_LOC"])

    # no layover - 0 minutes
    if "0 minute" in x.lower():
        return pd.Series([0.0, "NO_LOC"])

    # get hours and/or minutes
    hr = re.search(r"(\d+)\s*hr", x)
    mins = re.search(r"(\d+)\s*min", x)

    h = int(hr.group(1)) if hr else 0
    m = int(mins.group(1)) if mins else 0
    decimal = round(h+m/60, 2)

    # get location
    loc = re.search(r"\b([A-Z]{3})\b$", x)

    location = (
        loc.group(1)
        if loc
        else "NO_LOC"
    )

    return pd.Series([
        decimal,
        location
    ])


df_bom[
    ["Layover_Time", "Layover_Location"]
] = (
    df_bom["Layover_Time_Location"]
    .apply(split_layover)
)

df_bom.drop(
    columns=["Layover_Time_Location"],
    inplace=True
)
df_bom.head()

,Unnamed: 0,Departure_Time,Arrival_Time,Airline,Duration,Route,Stops,CO2_Emissions,Emissions_Change,Price,Class,Source,Destination,Layover_Time,Layover_Location
0,0,11:20 AM,3:45 PM,IndiGo,6.83,BOM–BLR,1,92,33.0,4383,economy,BOM,BLR,1.92,GOI
1,1,4:40 PM,10:15 PM,IndiGo,9.83,BOM–BLR,1,104,51.0,4383,economy,BOM,BLR,2.83,GOI
2,2,5:45 AM,10:55 AM,IndiGo,4.67,BOM–BLR,1,106,54.0,4514,economy,BOM,BLR,2.25,HYD
3,3,9:15 AM,1:35 PM,IndiGo,3.75,BOM–BLR,1,95,38.0,4514,economy,BOM,BLR,1.75,GOX
4,4,3:15 PM,7:05 PM,IndiGo,5.33,BOM–BLR,1,106,54.0,4514,economy,BOM,BLR,1.00,HYD


##### Departure and Arrival time to decimal

In [143]:
df_bom["Departure_Time"] = df_bom["Departure_Time"].str.strip()
df_bom["Arrival_Time"] = df_bom["Arrival_Time"].str.strip()
dep = pd.to_datetime(
        df_bom["Departure_Time"],
        format="%I:%M %p",
        errors="coerce"
    )
arr = pd.to_datetime(
        df_bom
["Arrival_Time"],
        format="%I:%M %p",
        errors="coerce"
    )
df_bom["Departure_Hour"] = dep.dt.hour + dep.dt.minute / 60
df_bom["Arrival_Hour"] = arr.dt.hour + arr.dt.minute / 60

df_bom.head()

,Unnamed: 0,Departure_Time,Arrival_Time,Airline,Duration,Route,Stops,CO2_Emissions,Emissions_Change,Price,Class,Source,Destination,Layover_Time,Layover_Location,Departure_Hour,Arrival_Hour
0,0,11:20 AM,3:45 PM,IndiGo,6.83,BOM–BLR,1,92,33.0,4383,economy,BOM,BLR,1.92,GOI,11.333333,15.750000
1,1,4:40 PM,10:15 PM,IndiGo,9.83,BOM–BLR,1,104,51.0,4383,economy,BOM,BLR,2.83,GOI,16.666667,22.250000
2,2,5:45 AM,10:55 AM,IndiGo,4.67,BOM–BLR,1,106,54.0,4514,economy,BOM,BLR,2.25,HYD,5.750000,10.916667
3,3,9:15 AM,1:35 PM,IndiGo,3.75,BOM–BLR,1,95,38.0,4514,economy,BOM,BLR,1.75,GOX,9.250000,13.583333
4,4,3:15 PM,7:05 PM,IndiGo,5.33,BOM–BLR,1,106,54.0,4514,economy,BOM,BLR,1.00,HYD,15.250000,19.083333


##### Remove unwanted columns

In [144]:
df_bom = df_bom.drop(
        columns=[
            "Route",
            "Unnamed: 0",
            "Departure_Time","Arrival_Time"
        ],
        errors="ignore"
    )

### Data (after cleaning and transformation):

In [145]:
df_bom.head()

,Airline,Duration,Stops,CO2_Emissions,Emissions_Change,Price,Class,Source,Destination,Layover_Time,Layover_Location,Departure_Hour,Arrival_Hour
0,IndiGo,6.83,1,92,33.0,4383,economy,BOM,BLR,1.92,GOI,11.333333,15.750000
1,IndiGo,9.83,1,104,51.0,4383,economy,BOM,BLR,2.83,GOI,16.666667,22.250000
2,IndiGo,4.67,1,106,54.0,4514,economy,BOM,BLR,2.25,HYD,5.750000,10.916667
3,IndiGo,3.75,1,95,38.0,4514,economy,BOM,BLR,1.75,GOX,9.250000,13.583333
4,IndiGo,5.33,1,106,54.0,4514,economy,BOM,BLR,1.00,HYD,15.250000,19.083333


#### Airlines:

In [149]:
print(df_bom["Airline"].unique())

['IndiGo' 'SriLankan' 'Etihad' 'Etihad, Akasa Air' 'Gulf Air'
 'Air India Express' 'IndiGo, Qatar Airways' 'Akasa Air' 'SpiceJet'
 'Air India' 'Star Air']


#### Sources:

In [152]:
print(df_bom["Source"].unique())

['BOM' 'BLR' 'MAA' 'DEL' 'IXE' 'HYD' 'CCU' 'COK' 'IXC' 'CJB' 'JAI' 'TRV'
 'IXR' 'PAT' 'GOX' 'IXG' 'GOI']


#### Destinations:

In [153]:
print(df_bom["Destination"].unique())

['BLR' 'BOM' 'MAA' 'DEL' 'IXE' 'HYD' 'CCU' 'COK' 'IXC' 'CJB' 'JAI' 'TRV'
 'IXR' 'PAT' 'GOX' 'GOI' 'IXG']


#### Number of flights:

In [146]:
len(df_bom)

462

#### Average price:

In [ ]:
print(df_bom["Price"].mean())

8451.89393939394


#### Average CO2 Emissions:

In [154]:
print(df_bom["CO2_Emissions"].mean())


106.44372294372295


#### Number of flights to (and fro) each destination:

In [148]:
# Flights from BOM 
from_bom = (
    df_bom[df_bom["Source"] == "BOM"]
    .groupby("Destination")
    .size()
    .rename("Flights from BOM")
)

# Flights to BOM 
to_bom = (
    df_bom[df_bom["Destination"] == "BOM"]
    .groupby("Source")
    .size()
    .rename("Flights to BOM")
)

# Combine 
flights_bom = pd.concat([to_bom, from_bom], axis=1).fillna(0).astype(int)

flights_bom.index.name = "Airport"
flights_bom = flights_bom.reset_index()

print(flights_bom)

   Airport  Flights to BOM  Flights from BOM
0      BLR              28                32
1      CCU              15                22
2      CJB               8                 6
3      COK              14                10
4      DEL              59                58
5      GOI               9                10
6      GOX               6                 6
7      HYD              24                21
8      IXC               9                 7
9      IXE               6                 8
10     IXG               1                 1
11     IXR               5                 4
12     JAI               6                 9
13     MAA              20                20
14     PAT               9                11
15     TRV               9                 9
